# Capítulo 3 – Modelo Base de Regresión Lineal Múltiple
## Sistema de Bicicletas Compartidas – Dataset `hour_prepared.csv`

En este capítulo ajustamos un **primer modelo base de regresión lineal múltiple** para explicar la demanda horaria de bicicletas (`cnt`) a partir de las variables preparadas en el Capítulo 2.

Trabajaremos con el dataset ya transformado y con variables dummies, almacenado en `../data/hour_prepared.csv`.


## 1. Carga de librerías y datos

Cargamos el dataset preparado y separamos la variable objetivo `cnt` de los predictores.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11


In [ ]:
# Carga del dataset preparado
data_path = "../data/hour_prepared.csv"
df_model = pd.read_csv(data_path)

df_model.head()

Verificamos dimensiones y presencia de valores faltantes:


In [ ]:
df_model.shape

In [ ]:
df_model.isna().sum().sum()

## 2. Definición de la variable objetivo y predictores

Definimos:

- Variable objetivo: `cnt`.
- Matriz de predictores: todas las demás columnas del dataset preparado.


In [ ]:
target_col = "cnt"
if target_col not in df_model.columns:
    raise ValueError("La columna 'cnt' no se encuentra en 'hour_prepared.csv'.")

y = df_model[target_col].copy()
X = df_model.drop(columns=[target_col]).copy()

X.shape, y.shape

## 3. Partición en conjuntos de entrenamiento y prueba

Dividimos los datos en:

- **Entrenamiento**: 80%
- **Prueba**: 20%

Utilizamos una semilla fija (`random_state=123`) para garantizar reproducibilidad.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

X_train.shape, X_test.shape

## 4. Ajuste del modelo OLS (mínimos cuadrados ordinarios)

Para utilizar `statsmodels`, añadimos un término de intercepto (`const`) a la matriz de diseño y ajustamos un modelo OLS sobre el conjunto de entrenamiento.


In [ ]:
# Añadimos constante a los predictores
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test, has_constant='add')

X_train_const.head()

In [ ]:
# Ajuste del modelo OLS en el conjunto de entrenamiento
ols = sm.OLS(y_train, X_train_const).fit()
ols

### 4.1. Resumen del modelo

Mostramos el resumen estándar de `statsmodels`, que incluye estimaciones de coeficientes, errores estándar, valores *t* y *p*-values.


In [ ]:
ols.summary()

### 4.2. Tabla de coeficientes en formato tabular

Construimos una tabla con:

- Coeficientes estimados (`coef`).
- Errores estándar (`std err`).
- Estadístico t (`t`).
- Valor-p (`P>|t|`).
- Intervalos de confianza al 95% (`IC 2.5%`, `IC 97.5%`).


In [ ]:
conf_int = ols.conf_int(alpha=0.05)

# Aseguramos que conf_int tenga nombres de columnas claros
conf_int.columns = ["IC 2.5%", "IC 97.5%"]

# Construimos la tabla de coeficientes como DataFrame
coef_table = pd.DataFrame({
    "coef": ols.params,
    "std err": ols.bse,
    "t": ols.tvalues,
    "P>|t|": ols.pvalues,
})

coef_table = pd.concat([coef_table, conf_int], axis=1)
coef_table.index.name = "Parámetro"
coef_table.round(4)

## 5. Métricas de desempeño del modelo base

Evaluamos el modelo tanto en el **conjunto de entrenamiento** como en el **conjunto de prueba**, utilizando:

- Error cuadrático medio (RMSE).
- Coeficiente de determinación (R²).


In [ ]:
# Predicciones en entrenamiento y prueba
y_pred_train = ols.predict(X_train_const)
y_pred_test = ols.predict(X_test_const)

# Cálculo de RMSE (sin usar el parámetro 'squared' para compatibilidad)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

# Cálculo de R²
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

metrics = pd.DataFrame({
    "RMSE": [rmse_train, rmse_test],
    "R2": [r2_train, r2_test],
}, index=["Train", "Test"])
metrics.round(4)

## 6. Gráficos básicos de ajuste

Como primer vistazo (antes de un capítulo de diagnóstico más profundo), graficamos:

- `y_train` vs `y_pred_train`.
- Histograma de residuales en entrenamiento.


In [ ]:
# Dispersión entre valores observados y predichos (entrenamiento)
fig, ax = plt.subplots()
ax.scatter(y_train, y_pred_train, alpha=0.3)
ax.set_xlabel("cnt observado (train)")
ax.set_ylabel("cnt predicho (train)")
ax.set_title("Valores observados vs predichos (train)")
plt.tight_layout()
plt.show()

In [ ]:
# Histograma de residuales en entrenamiento
residuals_train = y_train - y_pred_train
fig, ax = plt.subplots()
ax.hist(residuals_train, bins=30, edgecolor="black")
ax.set_title("Distribución de residuales (train)")
ax.set_xlabel("Residual")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

## 7. Resumen del capítulo

En este capítulo:

- Cargamos el dataset preparado `hour_prepared.csv`.
- Definimos la variable objetivo `cnt` y la matriz de predictores `X`.
- Dividimos los datos en conjuntos de entrenamiento y prueba.
- Ajustamos un modelo base de regresión lineal múltiple (OLS).
- Construimos una tabla de coeficientes con intervalos de confianza.
- Evaluamos el desempeño del modelo mediante RMSE y R².

En los siguientes capítulos profundizaremos en **diagnóstico del modelo**, detección de supuestos violados (normalidad, homocedasticidad, multicolinealidad) y posibles mejoras del modelo.
